In [5]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [6]:
load_dotenv()
MODEL = "openai/gpt-oss-20b"
DB_NAME = "vector_db"

In [7]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [8]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL, api_key=os.getenv("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")

In [9]:
retriever.invoke("Who is Avery?")

[Document(id='d134cba2-99d4-46ca-aad6-74a4659b9e1e', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [10]:
llm.invoke("Who is Avery?") # right now we do not pass any context to the LLM so LLM can't say anything about Avery.

AIMessage(content='I’m not sure which Avery you’re referring to—there are many people, characters, and even places named Avery. Could you let me know a bit more about the context? For example, are you asking about a public figure, a fictional character, a historical person, or something else? That’ll help me give you the most accurate answer.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 75, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 53, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.315882512, 'prompt_time': 0.00382793, 'completion_time': 0.146526239, 'total_time': 0.150354169}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'id': 'chatcmpl-f8b590d4-9de3-4cd5-a73c-cb4c7815252b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logpro

All togather.

In [11]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [12]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [13]:
answer_question("Who is Avery?", [])

'Avery Lancaster is the Co‑Founder and Chief Executive Officer (CEO) of **Insurellm**, based in San\u202fFrancisco, California. Born on March\u202f15,\u202f1985, she has steered the company since its inception in 2015, guiding it from a startup into a leading player in the insurance‑technology space.\n\n**Key highlights of Avery’s career at Insurellm:**\n\n| Year | Performance | Notable Achievements |\n|------|-------------|----------------------|\n| 2018 | Exceeds Expectations | Launched two new products that boosted market share |\n| 2019 | Meets Expectations | Maintained steady growth; addressed team culture |\n| 2020 | Below Expectations | Faced pandemic‑related challenges; later stabilized operations |\n| 2021 | Exceptional | Transitioned to remote work, adopted digital tools, and achieved record‑high customer satisfaction |\n| 2022 | Satisfactory | Rebuilt team dynamics and addressed employee concerns |\n| 2023 | Exceeds Expectations | Regained market leadership with personalized

In [14]:
gr.ChatInterface(answer_question).launch()

/home/arnab/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
